In [1]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"; os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"; os.environ["HF_HUB_DISABLE_XET"] = "1"
import numpy as np, pandas as pd, torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

JOB  = "aes"       # "aes" or "ell"
MODE = "full"     
MAXLEN, BATCH, EPOCHS, LR, HEAD_LR, GRAD_CKPT = 512, 8, 3, 2e-5, 1e-4, False

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
path = snapshot_download("microsoft/deberta-v3-base", allow_patterns=["*.json", "spm.model", "pytorch_model.bin"])
tok = AutoTokenizer.from_pretrained(path)
print("special ids (pad, cls, sep):", tok.pad_token_id, tok.cls_token_id, tok.sep_token_id, "(expected 0 1 2)")
assert (tok.pad_token_id, tok.cls_token_id, tok.sep_token_id) == (0, 1, 2)

csv = glob.glob(f"/kaggle/input/**/{JOB}_prepared.csv", recursive=True)[0]
df = pd.read_csv(csv)
TARGETS = ["score"] if JOB == "aes" else ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
print(JOB, df.shape, "| fold sizes:", df["fold"].value_counts().sort_index().tolist())
print("control characters:", int(df["text"].str.contains("[\x80-\x9f]").sum()), "(must be 0)")
lens = np.array([len(x) for x in tok(df["text"].sample(500, random_state=0).tolist(), add_special_tokens=False)["input_ids"]])
print(f"token length: median {np.median(lens):.0f}, 90th pct {np.percentile(lens, 90):.0f}, "
      f"share longer than {MAXLEN - 2}: {(lens > MAXLEN - 2).mean() * 100:.0f}%")

cfg = dict(model_path=path, data_csv=csv, targets=TARGETS, task=JOB, maxlen=MAXLEN, bs=BATCH, epochs=EPOCHS,
           lr=LR, head_lr=HEAD_LR, grad_ckpt=GRAD_CKPT, probe_steps=60 if MODE == "probe" else 0, out_dir="/kaggle/working")
json.dump(cfg, open("/kaggle/working/run_config.json", "w"))

GPUs: ['Tesla T4', 'Tesla T4']


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The tokenizer you are loading from '/root/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


special ids (pad, cls, sep): 0 1 2 (expected 0 1 2)
aes (17307, 5) | fold sizes: [3462, 3462, 3461, 3461, 3461]
control characters: 0 (must be 0)
token length: median 408, 90th pct 644, share longer than 510: 28%


In [2]:
%%writefile /kaggle/working/train_fold.py
import os, sys, json, time, random
import numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import cohen_kappa_score

cfg = json.load(open("/kaggle/working/run_config.json"))
FOLDS = [int(x) for x in sys.argv[1].split(",")]
T = cfg["targets"]
tok = AutoTokenizer.from_pretrained(cfg["model_path"])
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
assert (PAD, CLS, SEP) == (0, 1, 2)

def log(msg): print(msg, flush=True)

def wrap(ids):
    room = cfg["maxlen"] - 2
    if len(ids) > room:
        head = int(room * 0.75)
        ids = ids[:head] + ids[len(ids) - (room - head):]
    return [CLS] + ids + [SEP]

df = pd.read_csv(cfg["data_csv"])
enc = [wrap(ids) for ids in tok(df["text"].tolist(), add_special_tokens=False)["input_ids"]]
Y = df[T].values.astype(np.float32)
folds = df["fold"].values

def batch_tensors(idx):
    L = max(len(enc[i]) for i in idx)
    ids = torch.full((len(idx), L), PAD, dtype=torch.long)
    att = torch.zeros((len(idx), L), dtype=torch.long)
    for r, i in enumerate(idx):
        n = len(enc[i]); ids[r, :n] = torch.tensor(enc[i]); att[r, :n] = 1
    return ids.cuda(), att.cuda()

class Scorer(nn.Module):
    def __init__(self, mean_init):
        super().__init__()
        c = AutoConfig.from_pretrained(cfg["model_path"], hidden_dropout_prob=0.0, attention_probs_dropout_prob=0.0)
        self.backbone = AutoModel.from_pretrained(cfg["model_path"], config=c).float()
        if cfg["grad_ckpt"]:
            self.backbone.gradient_checkpointing_enable()
        self.head = nn.Linear(c.hidden_size, len(T))
        with torch.no_grad():
            self.head.weight.normal_(0, 0.02); self.head.bias.copy_(torch.tensor(mean_init))
    def forward(self, ids, att):
        h = self.backbone(input_ids=ids, attention_mask=att).last_hidden_state.float()
        m = att.unsqueeze(-1).float()
        return self.head((h * m).sum(1) / m.sum(1).clamp(min=1))

@torch.no_grad()
def predict(model, idx):
    model.eval()
    order = np.argsort([len(enc[i]) for i in idx])
    out = np.zeros((len(idx), len(T)), dtype=np.float32)
    for s in range(0, len(order), 32):
        sel = order[s:s + 32]
        ids, att = batch_tensors(idx[sel])
        with torch.autocast("cuda", dtype=torch.float16):
            out[sel] = model(ids, att).float().cpu().numpy()
    return out

def score(pred, y):
    rmse = float(np.sqrt(((pred - y) ** 2).mean()))
    if cfg["task"] == "aes":
        q = cohen_kappa_score(y[:, 0].astype(int), np.clip(np.rint(pred[:, 0]), 1, 6).astype(int), weights="quadratic")
        return q, rmse
    return -rmse, rmse

def run_fold(fold):
    out_csv = f"{cfg['out_dir']}/oof_{cfg['task']}_fold{fold}.csv"
    if os.path.exists(out_csv) and not cfg["probe_steps"]:
        log(f"fold {fold}: already done, skipping"); return
    random.seed(42 + fold); np.random.seed(42 + fold); torch.manual_seed(42 + fold); torch.cuda.manual_seed_all(42 + fold)
    tr, va = np.where(folds != fold)[0], np.where(folds == fold)[0]
    model = Scorer(Y[tr].mean(0)).cuda()
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": cfg["lr"]},
                             {"params": model.head.parameters(), "lr": cfg["head_lr"]}], weight_decay=0.01)
    spe = int(np.ceil(len(tr) / cfg["bs"])); total = spe * cfg["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda")
    step, t0, run_loss = 0, time.time(), []
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        perm = np.random.permutation(tr)
        for s in range(0, len(perm), cfg["bs"]):
            idx = perm[s:s + cfg["bs"]]
            ids, att = batch_tensors(idx)
            y = torch.tensor(Y[idx]).cuda()
            with torch.autocast("cuda", dtype=torch.float16):
                pred = model(ids, att)
            loss = nn.functional.mse_loss(pred.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss.append(loss.item()); step += 1
            if step % 100 == 0:
                log(f"fold {fold} step {step}/{total} loss {np.mean(run_loss[-100:]):.3f} | {(time.time() - t0) / 60:.1f} min")
            if cfg["probe_steps"] and step >= cfg["probe_steps"]:
                sps = (time.time() - t0) / step
                log(f"PROBE: {sps:.2f} s/step, {spe} steps/epoch -> about {sps * total / 60:.0f} min of training per fold "
                    f"(plus evaluation), peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB, "
                    f"last losses {np.mean(run_loss[-20:]):.3f}")
                return
        pred_va = predict(model, va)
        sc, rmse = score(pred_va, Y[va])
        log(f"fold {fold} epoch {ep}: {'QWK' if cfg['task'] == 'aes' else '-RMSE'} {sc:.4f} | RMSE {rmse:.4f} | {(time.time() - t0) / 60:.1f} min")
        if ep == 1 and ((cfg["task"] == "aes" and sc < 0.5) or (cfg["task"] != "aes" and rmse > 0.60)):
            log("NOT LEARNING after epoch 1, stopping the whole job"); sys.exit(1)
    torch.save({k: v.half() for k, v in model.state_dict().items()}, f"{cfg['out_dir']}/{cfg['task']}_fold{fold}.pt")
    pd.DataFrame(pred_va, columns=T).assign(row=va).to_csv(out_csv, index=False)
    log(f"fold {fold} done")

for f in FOLDS:
    try:
        run_fold(f)
    except Exception as e:
        log(f"fold {f} FAILED: {repr(e)[:300]}")
    torch.cuda.empty_cache()

Writing /kaggle/working/train_fold.py


In [3]:
if MODE == "probe":
    r = subprocess.run([sys.executable, "/kaggle/working/train_fold.py", "0"], capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-1200:])
else:
    n_gpu = torch.cuda.device_count()
    groups = [[0, 1, 2, 3, 4]] if n_gpu < 2 else [[0, 2, 4], [1, 3]]
    procs = []
    for g, folds in enumerate(groups):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append(subprocess.Popen([sys.executable, "/kaggle/working/train_fold.py", ",".join(map(str, folds))],
                                      env=env, stdout=open(f"/kaggle/working/log_gpu{g}.txt", "w"), stderr=subprocess.STDOUT))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(600)
        print(f"--- {(time.time() - t0) / 3600:.1f} h elapsed ---")
        for g in range(len(procs)):
            print(f"GPU{g}:", open(f"/kaggle/working/log_gpu{g}.txt").read().strip().split("\n")[-1])
    print("return codes:", [p.returncode for p in procs])
    for g in range(len(procs)):
        print(f"\n=== GPU{g} summary ===")
        print("\n".join(l for l in open(f"/kaggle/working/log_gpu{g}.txt").read().split("\n")
                        if "epoch" in l or "FAILED" in l or "done" in l))

--- 0.2 h elapsed ---
GPU0: fold 0 step 1000/5193 loss 0.430 | 8.7 min
GPU1: fold 1 step 1100/5193 loss 0.414 | 9.1 min
--- 0.3 h elapsed ---
GPU0: fold 0 step 2100/5193 loss 0.331 | 19.5 min
GPU1: fold 1 step 2200/5193 loss 0.346 | 19.0 min
--- 0.5 h elapsed ---
GPU0: fold 0 step 3200/5193 loss 0.298 | 29.3 min
GPU1: fold 1 step 3400/5193 loss 0.280 | 28.9 min
--- 0.7 h elapsed ---
GPU0: fold 0 step 4200/5193 loss 0.223 | 39.2 min
GPU1: fold 1 step 4500/5193 loss 0.192 | 38.9 min
--- 0.8 h elapsed ---
GPU0:   scaler.step(opt); scaler.update(); sched.step()
GPU1: fold 3 step 400/5193 loss 0.588 | 3.3 min
--- 1.0 h elapsed ---
GPU0: fold 2 step 1100/5193 loss 0.454 | 9.8 min
GPU1: fold 3 step 1700/5193 loss 0.378 | 14.0 min
--- 1.2 h elapsed ---
GPU0: fold 2 step 2100/5193 loss 0.349 | 19.7 min
GPU1: fold 3 step 2800/5193 loss 0.321 | 24.0 min
--- 1.3 h elapsed ---
GPU0: fold 2 step 3300/5193 loss 0.281 | 30.3 min
GPU1: fold 3 step 3900/5193 loss 0.217 | 33.9 min
--- 1.5 h elapsed ---
G

In [4]:
if MODE == "full" and glob.glob(f"/kaggle/working/oof_{JOB}_fold*.csv"):
    parts = [pd.read_csv(f) for f in sorted(glob.glob(f"/kaggle/working/oof_{JOB}_fold*.csv"))]
    print("folds finished:", len(parts), "of 5")
    oof = pd.concat(parts).sort_values("row").reset_index(drop=True)
    idc = "essay_id" if JOB == "aes" else "text_id"
    res = df.loc[oof["row"], [idc, "fold"] + TARGETS].reset_index(drop=True)
    for t in TARGETS:
        res[f"oof_{t}"] = oof[t].values
    res.to_csv(f"/kaggle/working/oof_{JOB}_deberta.csv", index=False)
    if JOB == "aes":
        from sklearn.metrics import cohen_kappa_score
        q = cohen_kappa_score(res["score"].astype(int), np.clip(np.rint(res["oof_score"]), 1, 6).astype(int), weights="quadratic")
        print(f"DeBERTa out-of-fold QWK: {q:.4f}   (LightGBM was 0.748, TF-IDF 0.716)")
    else:
        rm = {t: float(np.sqrt(((res[t] - res[f"oof_{t}"]) ** 2).mean())) for t in TARGETS}
        print(pd.Series(rm).round(3).to_string(), "\nmean RMSE:", round(float(np.mean(list(rm.values()))), 3), "(LightGBM was 0.512)")

folds finished: 5 of 5
DeBERTa out-of-fold QWK: 0.8061   (LightGBM was 0.748, TF-IDF 0.716)
